# 🌸 DAKSHU AI: One-Click Google Colab Fine-Tuning
### Fine-Tune Google Gemma 2 (2B) as Dakshu with QLoRA & Unsloth on a Free GPU

This notebook trains Dakshu with its friendly multilingual personality (English, Tamil, Tanglish) and specialized modules (Companion, Study Buddy, English Speaking Partner, Idea Explorer, Daily Motivation, Voice Companion).

In [ ]:
# Step 1: Install high-performance Unsloth & fine-tuning dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets transformers

In [ ]:
# Step 2: Load base Gemma 2 2B with 4-bit quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect: Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # 4bit quantization to fit easily in free Colab GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Step 3: Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Step 4: Load Dakshu's Fine-Tuning Dataset
from datasets import Dataset
import json

# Sample dataset pairs or upload train.jsonl from your repo
data_path = "train.jsonl"
try:
    with open(data_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    dataset = Dataset.from_list(rows)
    print(f"Loaded {len(dataset)} training examples!")
except FileNotFoundError:
    print("Please upload finetune/dataset/train.jsonl to Colab files!")

In [ ]:
# Step 5: Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "dakshu-outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
# Step 6: Test Dakshu Live in Colab
FastLanguageModel.for_inference(model)
prompt = "<start_of_turn>user\nVanakkam Dakshu! How are you doing today? Epdi irukeenga?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Step 7: Export to GGUF (q4_k_m) for Local Ollama!
# Automatically quantizes and saves dakshu-q4_k_m.gguf
model.save_pretrained_gguf("dakshu-gguf", tokenizer, quantization_method = "q4_k_m")
print("✅ Successfully exported dakshu-gguf/unsloth.Q4_K_M.gguf!")
print("Download this file and move it to your laptop: DAKSHU_AI/model/dakshu.gguf")